# Project FORESIGHT — EDA & Insight Memo (D2)
Exploring the analysis-ready dataset: demand patterns, seasonality,
top movers, dead stock, and data-quality findings.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/analysis_ready.csv", parse_dates=["date"])
print(df.shape)
df.head()

## 1. Data-quality recap
(Full log written by pipeline.py at data/processed/data_quality_log.txt)


In [ ]:
with open("../data/processed/data_quality_log.txt") as f:
    print(f.read())

## 2. Overall demand trend
Weekly total units sold across all SKUs — is demand growing, flat, or seasonal?


In [ ]:
weekly = df.set_index("date").resample("W")["units_sold"].sum()
plt.figure(figsize=(12, 4))
plt.plot(weekly.index, weekly.values)
plt.title("Total Weekly Units Sold — All SKUs")
plt.ylabel("Units sold")
plt.xlabel("Week")
plt.tight_layout()
plt.savefig("../reports/weekly_demand_trend.png", dpi=120)
plt.show()

## 3. Seasonality — demand by month and by season


In [ ]:
monthly = df.groupby(df["date"].dt.month)["units_sold"].sum()
plt.figure(figsize=(10, 4))
monthly.plot(kind="bar")
plt.title("Total Units Sold by Calendar Month")
plt.xlabel("Month")
plt.ylabel("Units sold")
plt.tight_layout()
plt.savefig("../reports/seasonality_by_month.png", dpi=120)
plt.show()

In [ ]:
season_sales = df.groupby("season")["units_sold"].sum().sort_values(ascending=False)
print(season_sales)

## 4. Top movers vs dead stock
Rank SKUs by total units sold over the whole history.


In [ ]:
sku_totals = df.groupby(["sku_id", "category"])["units_sold"].sum().reset_index()
sku_totals = sku_totals.sort_values("units_sold", ascending=False)

print("TOP 10 MOVERS")
print(sku_totals.head(10))

print("\nBOTTOM 10 (DEAD STOCK CANDIDATES)")
print(sku_totals.tail(10))

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(sku_totals.head(15)["sku_id"], sku_totals.head(15)["units_sold"])
plt.xticks(rotation=90)
plt.title("Top 15 SKUs by Total Units Sold")
plt.ylabel("Units sold")
plt.tight_layout()
plt.savefig("../reports/top_movers.png", dpi=120)
plt.show()

## 5. Category performance


In [ ]:
cat_perf = df.groupby("category").agg(
    total_units=("units_sold", "sum"),
    total_revenue=("revenue", "sum"),
    n_skus=("sku_id", "nunique"),
).sort_values("total_revenue", ascending=False)
print(cat_perf)

plt.figure(figsize=(8, 4))
cat_perf["total_revenue"].plot(kind="bar")
plt.title("Total Revenue by Category")
plt.ylabel("Revenue")
plt.tight_layout()
plt.savefig("../reports/category_revenue.png", dpi=120)
plt.show()

## 6. Promotion impact
Compare average daily units sold on promo days vs non-promo days.


In [ ]:
promo_effect = df.groupby("promo_flag")["units_sold"].mean()
print("Avg units/day — 0 = no promo, 1 = promo")
print(promo_effect)
lift_pct = (promo_effect[1] / promo_effect[0] - 1) * 100
print(f"\nPromotion lift: {lift_pct:.1f}% more units sold on promo days")

## 7. Holiday impact


In [ ]:
holiday_effect = df.groupby("is_holiday")["units_sold"].mean()
print(holiday_effect)

## 8. Day-of-week pattern


In [ ]:
df["dow"] = df["date"].dt.day_name()
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_sales = df.groupby("dow")["units_sold"].mean().reindex(dow_order)

plt.figure(figsize=(8, 4))
dow_sales.plot(kind="bar")
plt.title("Average Units Sold by Day of Week")
plt.ylabel("Avg units sold")
plt.tight_layout()
plt.savefig("../reports/day_of_week.png", dpi=120)
plt.show()

## 9. Current stock position snapshot
Quick look at inventory health as of the latest date in the data.


In [ ]:
latest_date = df["date"].max()
latest_inv = df[df["date"] == latest_date][
    ["sku_id", "category", "on_hand_units", "reorder_point", "lead_time_days"]
].drop_duplicates("sku_id")

latest_inv["below_reorder_point"] = latest_inv["on_hand_units"] < latest_inv["reorder_point"]
print(f"SKUs below reorder point as of {latest_date.date()}: "
      f"{latest_inv['below_reorder_point'].sum()} / {len(latest_inv)}")
latest_inv[latest_inv["below_reorder_point"]].head(10)

## 10. Key business insights (fill in after reviewing charts above)

Example structure — replace with your own numbers once you run this on
the real client data:

1. **Seasonality**: Demand peaks in <season/month>, roughly <X>% above the
   yearly average — reorder cycles should build in extra lead time before it.
2. **Top vs dead stock**: The top 10 SKUs account for <X>% of total units
   sold, while the bottom 10 sell almost nothing — these are markdown
   candidates.
3. **Promotions work**: Promo days show a <X>% lift in units sold —
   worth planning inventory ahead of known promo events.
4. **Stock risk today**: <X> SKUs are already below their reorder point —
   these need attention before the forecast model is even built.

Write these (with real numbers) into reports/eda_insight_memo.md for D2.
